# 02 · Model comparison and error analysis

scikit-learn token baseline vs fine-tuned DistilBERT, per-entity results, and the
inference backends (PyTorch / ONNX FP32 / ONNX INT8).

In [ ]:
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIG = ROOT / "reports" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
ner_m = json.loads((ROOT / "reports/ner_metrics.json").read_text())
sk_m = json.loads((ROOT / "reports/sk_token_baseline_metrics.json").read_text())
sec_m = json.loads((ROOT / "reports/secret_detector_metrics.json").read_text())
overall = pd.DataFrame({
    "sklearn baseline (LogReg, handcrafted features)": {k: sk_m["test"][k] for k in ("precision", "recall", "f1")},
    "DistilBERT fine-tuned (PyTorch)": {k: ner_m["test"][k] for k in ("precision", "recall", "f1")},
}).T
overall

In [ ]:
labels = [k for k in ner_m["test"]["per_entity"] if k.isupper()]
per = pd.DataFrame({
    "sklearn": {l: sk_m["test"]["per_entity"][l]["f1-score"] for l in labels},
    "DistilBERT": {l: ner_m["test"]["per_entity"][l]["f1-score"] for l in labels},
}).sort_values("DistilBERT")
ax = per.plot.barh(figsize=(7, 4.5), color=["#9aa5b4", "#2f5bea"])
ax.set_xlabel("entity-level F1 (test)"); ax.set_xlim(0, 1); ax.set_title("Per-entity F1")
plt.tight_layout(); plt.savefig(FIG / "per_entity_f1.png"); plt.show()
per.round(3)

## Calibration

Temperature scaling fitted on validation logits; ECE measured on test tokens.

In [ ]:
ner_m['calibration']

## Secret detector: in-distribution vs out-of-distribution

In [ ]:
rows = {}
for name, m in sec_m["models"].items():
    for part in ("test", "challenge_ood"):
        rows[(name, part)] = {k: m[part][k] for k in ("precision", "recall", "f1", "average_precision")}
pd.DataFrame(rows).T

The challenge set uses contexts and token formats that never appear in training
(e.g. GitLab `glpat-` tokens, keyword-free credentials). Remaining misses are
mostly 32-char hex secrets with no keyword nearby, which are indistinguishable
from MD5 hashes without context: a documented limitation.

## Inference backends

In [ ]:
bench = json.loads((ROOT / "reports/benchmark.json").read_text())
pd.DataFrame(bench["summary"]).T

In [ ]:
lat = pd.DataFrame([
    {"backend": b, "batch": l["batch"], "p95_ms": l["p95_ms"]}
    for b, r in bench["details"].items() for l in r["latency"]
]).pivot(index="batch", columns="backend", values="p95_ms")
ax = lat.plot.bar(figsize=(7, 3.5), color=["#2f5bea", "#9aa5b4", "#1f8a4c"])
ax.set_ylabel("p95 latency per call (ms)"); ax.set_title("CPU latency by backend and batch size")
plt.xticks(rotation=0); plt.tight_layout(); plt.savefig(FIG / "latency.png"); plt.show()
lat